# 04 — Held-out source-video evaluation

For each fold and seed, the encoder and every fitted read-out see only
outer-training sources. Ridge selection uses source-disjoint inner folds;
scaling, fitting, neutral-band calibration, and target scale are train-only.
Predictions for original and mirrored poses are then produced once on the
held-out sources with the same fitted read-out.

The lane factorial separates single-pass free prediction, two-pass odd/even
parity, and the zero-origin read-out constraint for both learned and paired
initial encoders. Measured visibility, acquisition, annotation, and combined
nuisance lanes are accompanied by a learned-plus-nuisance incremental lane.
The target-component oracle is self-consistency only, never a baseline.

Before any read-out, the notebook also compares `E(Mx)` with `S E(x)` for all
33 joints on exactly common-valid tokens. Its strict error is residual energy
divided by total representation energy: zero is exact, while unrelated
equal-energy representations are near one. `S` swaps joints but does not fit,
rotate, sign-flip, or align latent channels. This conservative diagnostic is
evaluated per checkpoint against its paired initialization. Mirrors remain
paired transformations, not new test cases.
Results remain about source videos, while folder labels remain annotations and
no clinical or unseen-person interpretation is permitted. Synthetic smoke
scores are pipeline diagnostics, not evidence.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display


def locate_suite_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for ancestor in (start, *start.parents):
        for candidate in (ancestor, ancestor / "neurips-laterality"):
            if (
                (candidate / "config" / "protocol.json").is_file()
                and (candidate / "laterality").is_dir()
            ):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate neurips-laterality from the current working directory."
    )


SUITE_ROOT = locate_suite_root()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))

from laterality.config import load_context

context = load_context(SUITE_ROOT / "config" / "protocol.json")
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")


def show_inline(figure):
    display(figure)
    plt.close(figure)


print(
    f"suite={SUITE_ROOT} profile={context.profile} "
    f"artifacts={context.artifact_root} protocol={context.protocol_digest[:12]}"
)

In [ ]:
from laterality.data import load_cohort
from laterality.evaluation import evaluate_selected
from laterality.splitting import load_splits
from laterality.visualization import evaluation_figure

cohort = load_cohort(context)
splits = load_splits(context, cohort)
evaluations = evaluate_selected(context, cohort, splits)

evaluation_summary = (
    evaluations.groupby(["variant", "fold", "seed", "lane"], as_index=False)
    .agg(test_sequences=("sequence_id", "size"), test_sources=("video_id", "nunique"))
    .sort_values(["variant", "fold", "seed", "lane"])
    .reset_index(drop=True)
)
representation_summary = (
    evaluations[
        evaluations["lane"]
        == context.protocol["evaluation"]["primary_lane"]
    ]
    .groupby(["variant", "seed"], as_index=False)
    .agg(
        learned_strict_error=(
            "learned_strict_equivariance_error",
            "mean",
        ),
        initial_strict_error=(
            "initial_strict_equivariance_error",
            "mean",
        ),
        minimum_common_tokens=(
            "learned_strict_equivariance_common_tokens",
            "min",
        ),
    )
    .sort_values(["variant", "seed"])
    .reset_index(drop=True)
)
display(evaluation_summary)
display(representation_summary)
show_inline(evaluation_figure(context, evaluations))